# Phase 3: Team-season features (transformation + reduction)

**Goal:** one row per team-season (1,170) describing **playing style** and **workload**, ready for clustering in
Phase 4. Every feature is a per-match rate or a season total, so leagues of 18 and 20 clubs and the shortened Ligue 1
2019-20 are comparable.

| Group | Features | Source |
|---|---|---|
| Pressing | PPDA (Σ passes allowed / Σ defensive actions), opponent PPDA | Understat |
| Attacking approach | deep completions for and against, npxG, npxGA (per match) | Understat |
| Physical intensity | fouls committed and suffered, cards (yellow + 2 × red), corners for and against (per match) | Football-Data |

Shooting (shots on target) is computed for the record but **not used**: it reflects team strength more than style,
is covered by npxG and deep completions, and Serie A's values are inflated in 2018-19 to 2020-21 (see `DECISIONS.md`).
| Workload | matches in all competitions, European matches, mean rest days, share of matches on < 4 days' rest, rest-day SD (gaps > 21 days left out) | TM fixtures |
| Rotation | share of the club's minutes played by its top 14 players | TM minutes (Phase 2) |

**Steps**
1. Team-season keys
2. Understat: pressing and attacking approach
3. Football-Data: physical intensity (and shots on target, not used)
4. TM fixtures: workload
5. TM minutes: rotation
6. One table, checked complete
7. How features differ by season and league
8. Correlations and redundant features
9. Standardise and save
10. Report

**Inputs:** `data/tables/` (`us_team_matches`, `fd_matches`, `tm_fixtures`), `data/01_entities/team_seasons.csv`,
`data/02_injuries/player_club_seasons.parquet`. **Outputs:** `data/03_features/`, `reports/03_features.md`.

In [1]:
import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))  # project root, for the analysis package
from analysis.config import DATA_DIR, REPORTS_DIR, phase_dir
from analysis.load import league_from_understat, read_table, season_key
from analysis.report import md_table

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)

PHASE = "03_features"
OUT_DIR = phase_dir(PHASE)
REDUNDANT_R = 0.9  # |r| above this marks a pair of features as redundant
TOP_N = 14  # rotation: share of minutes played by the club's top 14 players
SHORT_REST = 4  # a match on fewer than 4 days' rest counts as short rest
BREAK_DAYS = 21  # gaps longer than this are breaks (winter, COVID, World Cup 2022), not rest

LOG = []  # one row per preprocessing operation, for docs/PREPROCESSING_LOG.md


def log(step: str, operation: str, rows_in: int, rows_out: int) -> None:
    LOG.append({"step": step, "operation": operation, "rows in": rows_in, "rows out": rows_out})
    print(f"[{step}] {operation}: {rows_in:,} → {rows_out:,}")

## Step 1. Team-season keys

The 1,170 team-seasons from Phase 1, with each source's identity. Every feature table is joined to these keys.

In [2]:
keys = pd.read_csv(DATA_DIR / "01_entities" / "team_seasons.csv", dtype={"tm_club_id": str, "us_team_id": str})
assert len(keys) == 1170 and not keys.duplicated(["league", "season", "tm_club_id"]).any()
keys.head(3)

,league,season_key,season,tm_club_id,club,fd_name,us_team_id,us_name
0,Bundesliga,2014-15,2014,3,1.FC Köln,FC Koln,134,FC Cologne
1,Bundesliga,2014-15,2014,39,1.FSV Mainz 05,Mainz,125,Mainz 05
2,Bundesliga,2014-15,2014,15,Bayer 04 Leverkusen,Leverkusen,119,Bayer Leverkusen


## Step 2. Understat: pressing and attacking approach

Season PPDA is **Σ passes allowed / Σ defensive actions** over the team's league matches, not the mean of match
PPDAs (a low-pass match would otherwise weigh as much as a busy one). Lower PPDA = more intense pressing. *Opponent
PPDA* is the same for the opponents: low means the team is pressed hard, typical of teams that keep the ball.

In [3]:
um = read_table("us_team_matches")
num = ["ppda.att", "ppda.def", "ppda_allowed.att", "ppda_allowed.def", "deep", "deep_allowed", "npxG", "npxGA"]
um[num] = um[num].astype(float)
um["league"] = um.league.map(league_from_understat)
um["season"] = um.season.astype(int)
g = um.groupby(["league", "season", "team_id"])
understat = pd.DataFrame({
    "us_matches": g.size(),
    "ppda": g["ppda.att"].sum() / g["ppda.def"].sum(),
    "opp_ppda": g["ppda_allowed.att"].sum() / g["ppda_allowed.def"].sum(),
    "deep_pm": g.deep.mean(),
    "deep_against_pm": g.deep_allowed.mean(),
    "npxg_pm": g.npxG.mean(),
    "npxga_pm": g.npxGA.mean(),
}).reset_index().rename(columns={"team_id": "us_team_id"})
log("3.2", "Understat team-matches aggregated to team-seasons", len(um), len(understat))

[3.2] Understat team-matches aggregated to team-seasons: 43,178 → 1,170


In [4]:
# Check: one row per Understat team-season, plausible ranges.
understat.describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
season,1170.0,2019.48,3.45,2014.00,2016.00,2019.00,2022.00,2025.00
us_matches,1170.0,36.90,2.06,27.00,38.00,38.00,38.00,38.00
ppda,1170.0,10.89,2.31,5.50,9.17,10.71,12.44,19.00
opp_ppda,1170.0,10.83,2.92,4.07,8.84,10.27,12.28,26.31
deep_pm,1170.0,6.09,2.34,2.03,4.43,5.47,7.24,15.35
deep_against_pm,1170.0,6.09,1.52,2.44,4.94,6.06,7.13,11.68
npxg_pm,1170.0,1.28,0.37,0.63,1.00,1.20,1.47,2.89
npxga_pm,1170.0,1.28,0.29,0.52,1.08,1.27,1.48,2.40


## Step 3. Football-Data: physical intensity (and shots on target, not used)

Each match gives one row for the home team and one for the away team. *Fouls suffered* is the opponent's fouls
committed. Cards weigh a red as two yellows. Rates are means over the matches that have the statistic (3 matches
have none, Phase 0).

In [5]:
fd = read_table("fd_matches")
stats = ["HST", "AST", "HF", "AF", "HC", "AC", "HY", "AY", "HR", "AR"]
fd[stats] = fd[stats].apply(pd.to_numeric, errors="coerce")
fd["season"] = 2000 + fd.season_code.str[:2].astype(int)


def side(team: str, us: str, them: str) -> pd.DataFrame:
    return pd.DataFrame({
        "league": fd.league, "season": fd.season, "fd_name": fd[team],
        "sot_for": fd[f"{us}ST"], "sot_against": fd[f"{them}ST"],
        "fouls": fd[f"{us}F"], "fouled": fd[f"{them}F"],
        "corners_for": fd[f"{us}C"], "corners_against": fd[f"{them}C"],
        "cards": fd[f"{us}Y"] + 2 * fd[f"{us}R"],
    })


fd_long = pd.concat([side("HomeTeam", "H", "A"), side("AwayTeam", "A", "H")], ignore_index=True)
g = fd_long.groupby(["league", "season", "fd_name"])
football_data = g.agg(
    fd_matches=("sot_for", "size"),
    sot_for_pm=("sot_for", "mean"), sot_against_pm=("sot_against", "mean"),
    fouls_pm=("fouls", "mean"), fouled_pm=("fouled", "mean"), cards_pm=("cards", "mean"),
    corners_for_pm=("corners_for", "mean"), corners_against_pm=("corners_against", "mean"),
).reset_index()
log("3.3", "Football-Data matches to team-matches (home + away) to team-seasons", len(fd), len(football_data))

[3.3] Football-Data matches to team-matches (home + away) to team-seasons: 21,589 → 1,170


In [6]:
# Check: plausible ranges; Serie A shots on target by season (the total-shots definition changed in 2018-19).
display(football_data.describe().round(2).T)
football_data[football_data.league == "Serie A"].groupby("season").sot_for_pm.mean().round(2).to_frame().T

,count,mean,std,min,25%,50%,75%,max
season,1170.0,2019.48,3.45,2014.00,2016.00,2019.00,2022.00,2025.00
fd_matches,1170.0,36.90,2.06,27.00,38.00,38.00,38.00,38.00
sot_for_pm,1170.0,4.40,1.04,2.34,3.66,4.21,4.92,8.50
sot_against_pm,1170.0,4.40,0.86,2.18,3.84,4.39,4.94,7.47
fouls_pm,1170.0,12.52,1.87,7.55,11.16,12.47,13.76,18.55
fouled_pm,1170.0,12.52,1.79,7.68,11.22,12.50,13.71,18.92
cards_pm,1170.0,2.23,0.51,1.00,1.87,2.18,2.56,3.97
corners_for_pm,1170.0,4.92,0.87,2.97,4.32,4.82,5.45,8.44
corners_against_pm,1170.0,4.92,0.88,2.00,4.33,4.89,5.53,7.50


season,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
sot_for_pm,4.32,4.16,4.67,4.37,5.24,5.71,5.3,4.44,4.09,4.15,4.04,4.03


## Step 4. TM fixtures: workload

All official matches of each club-season (league, domestic cups, super cups, European and other international club
competitions, relegation playoffs), from TM fixtures. **Rest days** = days since the club's previous match in the
same season; the first match of a season has none. Gaps of more than 21 days are breaks (winter break, COVID
suspension, World Cup 2022 stop; 0.7% of gaps) and are left out of the rest features; international breaks stay in. *European matches* are UEFA and FIFA club competitions
(competition types 10 and 13, qualifiers included).

In [7]:
fx = read_table("tm_fixtures")
fx = pd.DataFrame({
    "tm_club_id": fx.club_id, "game_id": fx.game_id, "season": fx["game.seasonId"].astype(int),
    "type": fx["game.competitionTypeId"].astype(int), "date": pd.to_datetime(fx["game.date.dateTimeUTC"].str[:10]),
}).drop_duplicates(["tm_club_id", "game_id"]).sort_values(["tm_club_id", "season", "date"])
fx["rest_days"] = fx.groupby(["tm_club_id", "season"]).date.diff().dt.days
breaks = fx.rest_days > BREAK_DAYS
print(f"{breaks.sum():,} gaps of more than {BREAK_DAYS} days left out of the rest features ({breaks.mean():.1%} of gaps)")
fx.loc[breaks, "rest_days"] = np.nan
g = fx.groupby(["tm_club_id", "season"])
workload = pd.DataFrame({
    "matches_all": g.size(),
    "matches_europe": g.type.apply(lambda t: t.isin([10, 13]).sum()),
    "rest_mean": g.rest_days.mean(),
    "short_rest_share": g.rest_days.apply(lambda r: (r.dropna() < SHORT_REST).mean()),
    "rest_sd": g.rest_days.std(),
}).reset_index()
log("3.4", "TM club fixtures to club-season workload", len(fx), len(workload))

369 gaps of more than 21 days left out of the rest features (0.7% of gaps)


[3.4] TM club fixtures to club-season workload: 52,275 → 1,170


In [8]:
# Check: ranges, and no two matches of a club on one day.
assert (fx.rest_days.dropna() > 0).all()
workload.describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
season,1170.0,2019.48,3.45,2014.00,2016.00,2019.00,2022.00,2025.00
matches_all,1170.0,44.68,6.88,30.00,40.00,42.00,49.00,68.00
matches_europe,1170.0,3.68,5.37,0.00,0.00,0.00,8.00,24.00
rest_mean,1170.0,6.31,0.86,4.15,5.59,6.50,7.00,8.26
short_rest_share,1170.0,0.21,0.11,0.00,0.12,0.20,0.29,0.52
rest_sd,1170.0,3.23,0.38,2.07,2.99,3.23,3.46,4.61


## Step 5. TM minutes: rotation

From Phase 2's membership table (minutes in all competitions for each player at each club-season): the share of the
club's minutes played by its **top 14 players**. High = a fixed core plays most minutes; low = heavy rotation.
The Herfindahl index (sum of squared minute shares) is computed as a check on the choice.

In [9]:
members = pd.read_parquet(DATA_DIR / "02_injuries" / "player_club_seasons.parquet")


def concentration(minutes: pd.Series) -> pd.Series:
    m = minutes.sort_values(ascending=False)
    share = m / m.sum()
    return pd.Series({"top14_share": share.head(TOP_N).sum(), "minutes_hhi": (share ** 2).sum(), "players_used": int((m > 0).sum())})


rotation = members.groupby(["club_id", "season"]).minutes.apply(concentration).unstack().reset_index()
rotation = rotation.rename(columns={"club_id": "tm_club_id"})
log("3.5", "player-club-season minutes to club-season rotation", len(members), len(rotation))
print(f"top-14 share vs HHI: r = {rotation.top14_share.corr(rotation.minutes_hhi):.2f}")
rotation.describe().round(3).T

[3.5] player-club-season minutes to club-season rotation: 36,525 → 1,170
top-14 share vs HHI: r = 0.98


,count,mean,std,min,25%,50%,75%,max
season,1170.0,2019.477,3.447,2014.000,2016.00,2019.000,2022.000,2025.000
top14_share,1170.0,0.801,0.050,0.598,0.77,0.802,0.835,0.952
minutes_hhi,1170.0,0.053,0.005,0.037,0.05,0.053,0.057,0.068
players_used,1170.0,30.009,3.680,19.000,27.00,30.000,32.000,48.000


## Step 6. One table

All feature groups joined to the 1,170 keys. Every team-season must have every feature, and each source's match
count must agree with the expected league schedule.

In [10]:
features = (keys
            .merge(understat, on=["league", "season", "us_team_id"], how="left")
            .merge(football_data, on=["league", "season", "fd_name"], how="left")
            .merge(workload, on=["tm_club_id", "season"], how="left")
            .merge(rotation, on=["tm_club_id", "season"], how="left"))
FEATURES = {
    "pressing": ["ppda", "opp_ppda"],
    "attacking approach": ["deep_pm", "deep_against_pm", "npxg_pm", "npxga_pm"],
    "physical intensity": ["fouls_pm", "fouled_pm", "cards_pm", "corners_for_pm", "corners_against_pm"],
    "workload": ["matches_all", "matches_europe", "rest_mean", "short_rest_share", "rest_sd"],
    "rotation": ["top14_share"],
}
ALL = [f for group in FEATURES.values() for f in group]
assert len(features) == 1170
missing = features[ALL].isna().sum()
assert not missing.any(), missing[missing > 0]
assert (features.us_matches == features.fd_matches).all()
log("3.6", "feature groups joined to team-seasons", len(keys), len(features))
print(len(ALL), "features, none missing;", "league matches agree between Understat and Football-Data")

[3.6] feature groups joined to team-seasons: 1,170 → 1,170
17 features, none missing; league matches agree between Understat and Football-Data


## Step 7. How features differ by season and league

Season means show how football changed over the period; league means show how leagues differ. Both are part of
what clustering should see, which is why the features are standardised over all team-seasons together (step 9).

In [11]:
by_season = features.groupby("season_key")[ALL].mean()
by_league = features.groupby("league")[ALL].mean()
display(by_season.round(2))
display(by_league.round(2))

,ppda,opp_ppda,deep_pm,deep_against_pm,npxg_pm,npxga_pm,fouls_pm,fouled_pm,cards_pm,corners_for_pm,corners_against_pm,matches_all,matches_europe,rest_mean,short_rest_share,rest_sd,top14_share
season_key,,,,,,,,,,,,,,,,,
2014-15,9.12,9.03,5.44,5.44,1.17,1.17,13.93,13.93,2.30,5.04,5.04,44.92,3.36,6.39,0.20,3.23,0.81
2015-16,9.13,9.03,5.69,5.69,1.21,1.21,13.47,13.47,2.33,5.05,5.05,45.20,3.53,6.30,0.21,3.23,0.80
2016-17,9.86,9.77,5.74,5.74,1.19,1.19,13.20,13.20,2.22,4.96,4.96,44.85,3.21,6.42,0.20,3.33,0.80
2017-18,10.53,10.51,5.93,5.93,1.20,1.20,12.67,12.67,2.12,4.98,4.98,45.13,3.43,6.36,0.21,3.33,0.80
2018-19,10.91,10.90,6.05,6.05,1.29,1.29,12.26,12.26,2.22,5.02,5.02,45.00,3.45,6.37,0.20,3.26,0.81
2019-20,10.89,10.82,5.82,5.81,1.27,1.27,12.60,12.60,2.35,5.04,5.04,42.80,3.24,5.93,0.27,3.28,0.81
2020-21,11.44,11.39,6.04,6.04,1.23,1.23,12.61,12.61,2.10,4.73,4.73,44.24,3.31,5.96,0.26,3.14,0.80
2021-22,11.21,11.24,6.27,6.27,1.29,1.29,12.03,12.03,2.27,4.89,4.89,44.61,3.55,6.54,0.21,3.60,0.79
2022-23,11.45,11.40,6.51,6.51,1.32,1.32,12.05,12.05,2.27,4.80,4.80,44.73,3.69,5.91,0.20,2.60,0.80


,ppda,opp_ppda,deep_pm,deep_against_pm,npxg_pm,npxga_pm,fouls_pm,fouled_pm,cards_pm,corners_for_pm,corners_against_pm,matches_all,matches_europe,rest_mean,short_rest_share,rest_sd,top14_share
league,,,,,,,,,,,,,,,,,
Bundesliga,11.28,11.19,6.47,6.47,1.39,1.39,12.40,12.39,2.03,4.82,4.82,41.08,3.86,6.51,0.18,3.11,0.82
La Liga,9.92,9.83,5.69,5.69,1.24,1.24,13.35,13.35,2.71,4.78,4.78,46.51,4.07,6.08,0.25,3.27,0.78
Ligue 1,10.99,10.91,5.41,5.41,1.22,1.22,12.70,12.70,2.10,4.77,4.77,43.15,2.98,6.62,0.17,3.27,0.81
Premier League,11.33,11.33,6.87,6.87,1.33,1.33,10.79,10.79,1.86,5.21,5.21,47.94,4.00,5.96,0.26,3.23,0.80
Serie A,10.95,10.93,6.03,6.03,1.24,1.24,13.37,13.37,2.45,4.99,4.99,44.32,3.51,6.39,0.17,3.23,0.79


## Step 8. Correlations and redundant features

Pairs with |r| > 0.9 carry nearly the same information and would count twice in the distance K-means uses. For
each such pair one feature is dropped; the choice is written next to `DROP` in step 9.

In [12]:
corr = features[ALL].corr()
pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().rename("r").reset_index()
         .rename(columns={"level_0": "feature_a", "level_1": "feature_b"}))
pairs["abs_r"] = pairs.r.abs()
strong = pairs.sort_values("abs_r", ascending=False)
redundant = strong[strong.abs_r > REDUNDANT_R]
print(f"{len(redundant)} pairs with |r| > {REDUNDANT_R}")
display(redundant.round(3))
strong.head(12).round(3)

1 pairs with |r| > 0.9


,feature_a,feature_b,r,abs_r
235,rest_mean,short_rest_share,-0.902,0.902


,feature_a,feature_b,r,abs_r
235,rest_mean,short_rest_share,-0.902,0.902
200,matches_all,rest_mean,-0.894,0.894
199,matches_all,matches_europe,0.882,0.882
38,deep_pm,npxg_pm,0.881,0.881
201,matches_all,short_rest_share,0.834,0.834
217,matches_europe,rest_mean,-0.833,0.833
56,deep_against_pm,npxga_pm,0.828,0.828
218,matches_europe,short_rest_share,0.769,0.769
19,opp_ppda,deep_pm,0.751,0.751
21,opp_ppda,npxg_pm,0.736,0.736


## Step 9. Standardise and save

Z-scores over **all** team-seasons together (open decision 4), so that changes over time and differences between
leagues stay visible to the clustering. Raw values and z-scores are both saved.

In [13]:
# For each redundant pair, the feature to drop (see DECISIONS.md).
# rest_mean vs short_rest_share (r = -0.90): matches on < 4 days' rest are the congestion measure linked to injury
# risk (Dupont et al. 2010), and rest_mean is also close to matches_all (r = -0.89).
DROP = ["rest_mean"]
unresolved = redundant[~(redundant.feature_a.isin(DROP) | redundant.feature_b.isin(DROP))]
assert unresolved.empty, f"redundant pairs without a decision: {unresolved[['feature_a', 'feature_b']].values.tolist()}"
KEEP = [f for f in ALL if f not in DROP]
z = (features[KEEP] - features[KEEP].mean()) / features[KEEP].std(ddof=0)
out = features[["league", "season", "season_key", "tm_club_id", "club"] + ALL].join(z.add_prefix("z_"))
out.to_parquet(OUT_DIR / "team_season_features.parquet", index=False)
log("3.9", f"standardised {len(KEEP)} features over all team-seasons (dropped: {', '.join(DROP) or 'none'})", len(features), len(out))
assert np.allclose(z.mean(), 0) and np.allclose(z.std(ddof=0), 1)
print("kept:", KEEP)

[3.9] standardised 16 features over all team-seasons (dropped: rest_mean): 1,170 → 1,170
kept: ['ppda', 'opp_ppda', 'deep_pm', 'deep_against_pm', 'npxg_pm', 'npxga_pm', 'fouls_pm', 'fouled_pm', 'cards_pm', 'corners_for_pm', 'corners_against_pm', 'matches_all', 'matches_europe', 'short_rest_share', 'rest_sd', 'top14_share']


## Step 10. Report

In [14]:
log_table = pd.DataFrame(LOG)
log_table.to_csv(OUT_DIR / "preprocessing_counts.csv", index=False)
groups = pd.DataFrame([(g, f, "dropped (redundant)" if f in DROP else "kept") for g, fs in FEATURES.items() for f in fs],
                      columns=["group", "feature", "status"])
ranges = features[ALL].describe().T[["mean", "std", "min", "max"]].round(2)
text = [
    "# Phase 3: Team-season features",
    "",
    f"Generated {date.today().isoformat()} by `notebooks/03_features.ipynb`, where each step is shown with its checks.",
    "",
    "## Key findings",
    "",
    f"- **{len(out):,} team-seasons, {len(ALL)} features, none missing.** League match counts agree between Understat and Football-Data.",
    f"- **Redundant pairs (|r| > {REDUNDANT_R}):** {len(redundant)}" + (f"; dropped {', '.join(DROP)}." if DROP else "."),
    f"- **{len(KEEP)} features** go into clustering, standardised over all team-seasons.",
    "",
    "## 1. Features",
    "",
    md_table(groups, index=False),
    "",
    "## 2. Ranges (raw units)",
    "",
    md_table(ranges.rename_axis("feature")),
    "",
    "## 3. Strongest correlations",
    "",
    md_table(strong.head(12)[["feature_a", "feature_b", "r"]].round(3), index=False),
    "",
    "## 4. Season means",
    "",
    md_table(by_season.round(2).T.rename_axis("feature")),
    "",
    "## 5. League means",
    "",
    md_table(by_league.round(2).T.rename_axis("feature")),
    "",
    "## 6. Steps and row counts",
    "",
    md_table(log_table, index=False),
    "",
    "## 7. Outputs",
    "",
    f"- `data/{PHASE}/team_season_features.parquet`: raw features and z-scores (`z_*`, the {len(KEEP)} kept features).",
    "",
]
(REPORTS_DIR / "03_features.md").write_text("\n".join(text), encoding="utf-8")
print("wrote", REPORTS_DIR / "03_features.md")

wrote /Users/mohammadshweiki/Downloads/coding-projects/football-injuries/reports/03_features.md
